# DeepAR: Вероятностное прогнозирование с RNN

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/23_deepar.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q gluonts torch pandas numpy matplotlib

## Подготовка данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём синтетические данные
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

# Несколько рядов
series_data = []
for i in range(10):
    y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi) + np.random.randn(365) * 5
    series_data.append(pd.DataFrame({
        'unique_id': f'series_{i}',
        'ds': dates,
        'y': np.maximum(y, 0)  # DeepAR с NegativeBinomial требует неотрицательных значений
    }))

train_df = pd.concat(series_data, ignore_index=True)
print(f"Всего рядов: {train_df['unique_id'].nunique()}")
print(train_df.head())

## DeepAR с GluonTS

In [ ]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.model.deepar import DeepAREstimator
from gluonts.torch.distributions import NegativeBinomialOutput, StudentTOutput

# Создаём dataset
dataset = PandasDataset.from_long_dataframe(
    train_df,
    item_id='unique_id',
    timestamp='ds',
    target='y',
    freq='D'
)

# Конфигурация DeepAR
estimator = DeepAREstimator(
    freq='D',
    prediction_length=16,
    context_length=64,                    # длина контекста
    
    # Архитектура
    num_layers=2,                         # количество LSTM слоёв
    hidden_size=40,                       # размер скрытого состояния
    dropout_rate=0.1,
    
    # Распределение выхода
    distr_output=StudentTOutput(),        # для данных с выбросами
    
    # Лаги для сезонности
    lags_seq=[1, 7, 14, 21, 28],          # вчера, неделю назад, ...
    
    # Обучение
    lr=1e-3,
    batch_size=32,
    num_batches_per_epoch=50,
    trainer_kwargs={'max_epochs': 20}
)

# Обучаем
print("Обучение модели...")
predictor = estimator.train(dataset)
print("Модель обучена!")

## Генерация прогнозов

In [ ]:
from gluonts.evaluation import make_evaluation_predictions

forecast_it, ts_it = make_evaluation_predictions(
    dataset=dataset,
    predictor=predictor,
    num_samples=100  # Monte Carlo samples
)

forecasts = list(forecast_it)
tss = list(ts_it)

# Доступ к результатам
for forecast in forecasts[:3]:
    print(f"Item: {forecast.item_id}")
    print(f"  Mean: {forecast.mean[:5]}")
    print(f"  Median: {forecast.median[:5]}")
    print(f"  P10: {forecast.quantile(0.1)[:5]}")
    print(f"  P90: {forecast.quantile(0.9)[:5]}")
    print()

## Визуализация вероятностного прогноза

In [ ]:
import matplotlib.pyplot as plt

def plot_probabilistic_forecast(ts, forecast, title=''):
    """
    Визуализация истории и вероятностного прогноза.
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # История
    history = ts[-100:].to_timestamp()
    ax.plot(history.index, history.values, color='black', label='История')
    
    # Прогноз
    forecast_index = pd.date_range(
        start=history.index[-1] + pd.Timedelta(days=1),
        periods=len(forecast.mean),
        freq='D'
    )
    
    # Медиана
    ax.plot(forecast_index, forecast.median, color='blue', label='Медиана')
    
    # Интервалы
    ax.fill_between(
        forecast_index,
        forecast.quantile(0.1),
        forecast.quantile(0.9),
        alpha=0.3,
        color='blue',
        label='80% интервал'
    )
    
    ax.fill_between(
        forecast_index,
        forecast.quantile(0.25),
        forecast.quantile(0.75),
        alpha=0.5,
        color='blue',
        label='50% интервал'
    )
    
    ax.legend()
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Визуализируем первый прогноз
plot_probabilistic_forecast(tss[0], forecasts[0], title=f'DeepAR: прогноз для {forecasts[0].item_id}')

## Визуализация нескольких рядов

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

for i, ax in enumerate(axes):
    ts = tss[i]
    forecast = forecasts[i]
    
    # История
    history = ts[-50:].to_timestamp()
    ax.plot(history.index, history.values, color='black', label='История')
    
    # Прогноз
    forecast_index = pd.date_range(
        start=history.index[-1] + pd.Timedelta(days=1),
        periods=len(forecast.mean),
        freq='D'
    )
    
    ax.plot(forecast_index, forecast.median, color='blue', label='Медиана')
    ax.fill_between(
        forecast_index,
        forecast.quantile(0.1),
        forecast.quantile(0.9),
        alpha=0.3,
        color='blue',
        label='80% интервал'
    )
    
    ax.set_title(f'{forecast.item_id}')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()